# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishigupgta1234-ux/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My rule: Prioritize pages that have not been updated for a long time but are still receiving meaningful search visibility. These pages are good candidates for a content refresh because they may still have traffic potential.

Reason code:
STALE_VISIBLE — the page is stale (days_since_last_update >= 180) and still visible (impressions_90d >= 500).

Action:
REFRESH — review the page and consider updating its content.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the ranked queue

import pandas as pd
import os

url = "https://raw.githubusercontent.com/ishigupgta1234-ux/flyrank-ml-internship/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Signals used in my rule
df["stale"] = df["days_since_last_update"] >= 180
df["visible"] = df["impressions_90d"] >= 500

# Give points to pages that match the rule
df["score"] = 0

df.loc[df["stale"], "score"] += 2
df.loc[df["visible"], "score"] += 1

# Reason code
df["reason_code"] = "NO_FLAG"
df.loc[df["stale"] & df["visible"], "reason_code"] = "STALE_VISIBLE"

# Action
df["action"] = "NO_ACTION"
df.loc[df["score"] == 1, "action"] = "MONITOR"
df.loc[df["score"] == 2, "action"] = "REVIEW"
df.loc[df["score"] == 3, "action"] = "REFRESH"

# Rank the pages
ranked = df.sort_values(
    ["score", "days_since_last_update", "impressions_90d"],
    ascending=False
).reset_index(drop=True)

ranked["rank"] = ranked.index + 1

# Keep only the columns needed for the queue
queue = ranked[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
]

# Save the output
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows written:", len(queue))
print(queue.head(10))

Rows written: 30000
   rank            content_id  score    reason_code   action  \
0     1  content_7f116ae1f6f5      3  STALE_VISIBLE  REFRESH   
1     2  content_72496874f806      3  STALE_VISIBLE  REFRESH   
2     3  content_cf56e2e2e282      3  STALE_VISIBLE  REFRESH   
3     4  content_7368877ea310      3  STALE_VISIBLE  REFRESH   
4     5  content_1bfaa38ff26c      3  STALE_VISIBLE  REFRESH   
5     6  content_5feee3994adb      3  STALE_VISIBLE  REFRESH   
6     7  content_b16bd7307b39      3  STALE_VISIBLE  REFRESH   
7     8  content_fe16a55cd13d      3  STALE_VISIBLE  REFRESH   
8     9  content_ecb6215e79fd      3  STALE_VISIBLE  REFRESH   
9    10  content_bdbec75c1148      3  STALE_VISIBLE  REFRESH   

   days_since_last_update  impressions_90d  
0                     301              954  
1                     301              821  
2                     194            61678  
3                     194            59472  
4                     194            25715  
5    

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Top-20 review

top20 = ranked.head(20)

for i, row in top20.iterrows():

    if row["score"] == 3:
        confidence = "High"
        wrong = "The page may be intentionally kept old or the impressions may be temporary."
    elif row["score"] == 2:
        confidence = "Medium"
        wrong = "The page is old, but it may not need a refresh."
    elif row["score"] == 1:
        confidence = "Low"
        wrong = "The page has visibility, but there may be no real refresh opportunity."
    else:
        confidence = "Low"
        wrong = "The rule may have missed another useful signal."

    print(
        f"{int(row['rank'])}. {row['content_id']} | "
        f"Action: {row['action']} | "
        f"Reason: {row['reason_code']} | "
        f"Confidence: {confidence} | "
        f"Wrong if: {wrong}"
    )



1. content_7f116ae1f6f5 | Action: REFRESH | Reason: STALE_VISIBLE | Confidence: High | Wrong if: The page may be intentionally kept old or the impressions may be temporary.
2. content_72496874f806 | Action: REFRESH | Reason: STALE_VISIBLE | Confidence: High | Wrong if: The page may be intentionally kept old or the impressions may be temporary.
3. content_cf56e2e2e282 | Action: REFRESH | Reason: STALE_VISIBLE | Confidence: High | Wrong if: The page may be intentionally kept old or the impressions may be temporary.
4. content_7368877ea310 | Action: REFRESH | Reason: STALE_VISIBLE | Confidence: High | Wrong if: The page may be intentionally kept old or the impressions may be temporary.
5. content_1bfaa38ff26c | Action: REFRESH | Reason: STALE_VISIBLE | Confidence: High | Wrong if: The page may be intentionally kept old or the impressions may be temporary.
6. content_5feee3994adb | Action: REFRESH | Reason: STALE_VISIBLE | Confidence: High | Wrong if: The page may be intentionally kept old

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks + leakage check

print("Weak picks:")
print()

# Look at the lowest scoring pages in the top 20
weak_picks = ranked.head(20).tail(5)

for _, row in weak_picks.iterrows():
    print(
        f"{int(row['rank'])}. {row['content_id']} - "
        f"Score: {row['score']} - "
        f"Action: {row['action']}"
    )

print()
print("Why these may be weak:")
print("Some pages can score high because they are old and visible,")
print("but that does not always mean that the content actually needs")
print("a refresh. A page may be intentionally evergreen or its traffic")
print("may be temporary.")

print()
print("Leakage check:")

# The baseline only uses these two signals
used_columns = [
    "days_since_last_update",
    "impressions_90d"
]

print("Signals used:", used_columns)
print("No product flags were used.")
print("No future-window or label-derived features were used.")
print("The score is based only on the current page signals.")

Weak picks:

16. content_6226ee6adc91 - Score: 3 - Action: REFRESH
17. content_074ba6ead17b - Score: 3 - Action: REFRESH
18. content_55a5b1c46474 - Score: 2 - Action: REVIEW
19. content_f6fdf87348f6 - Score: 2 - Action: REVIEW
20. content_3f3576c295f5 - Score: 2 - Action: REVIEW

Why these may be weak:
Some pages can score high because they are old and visible,
but that does not always mean that the content actually needs
a refresh. A page may be intentionally evergreen or its traffic
may be temporary.

Leakage check:
Signals used: ['days_since_last_update', 'impressions_90d']
No product flags were used.
No future-window or label-derived features were used.
The score is based only on the current page signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.